# Huấn luyện RT-DETR cho nhận diện biển báo (Zalo AI 2020)
Notebook này được tinh chỉnh riêng để chạy trên nền tảng **Kaggle**. Dữ liệu sẽ được đọc trực tiếp từ `/kaggle/input/` và mọi kết quả (trọng số, biểu đồ) sẽ được lưu an toàn tại `/kaggle/working/` để tải về.

In [ ]:
# 1. CHUẨN BỊ THƯ MỤC VÀ CHIA DATA 80/20 TỪ KAGGLE INPUT
import os
import json
import glob
import shutil
import random
from tqdm import tqdm

# Tìm file JSON tự động trong ổ đĩa ảo của Kaggle
json_paths = glob.glob('/kaggle/input/**/train_traffic_sign_dataset.json', recursive=True)
if not json_paths:
    raise FileNotFoundError("Không tìm thấy file JSON. Bạn đã Add Data vào Kaggle chưa?")

json_path = json_paths[0]
image_dir = os.path.dirname(json_path).replace('traffic_train', 'traffic_train/images')

if not os.path.exists(image_dir):
    img_dirs = glob.glob('/kaggle/input/**/traffic_train/images', recursive=True)
    if img_dirs:
        image_dir = img_dirs[0]

# Khởi tạo khu vực làm việc (Output) trên Kaggle
dataset_dir = '/kaggle/working/dataset'

for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/{split}/images', exist_ok=True)
    os.makedirs(f'{dataset_dir}/{split}/labels', exist_ok=True)

print("Đang đọc dữ liệu JSON...")
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_info = {img['id']: img for img in data['images']}

img_to_anns = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in img_to_anns:
        img_to_anns[img_id] = []
    img_to_anns[img_id].append(ann)

image_ids = list(images_info.keys())
random.seed(42)
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.8)
train_ids = image_ids[:split_idx]
val_ids = image_ids[split_idx:]

print(f"Tổng số ảnh: {len(image_ids)}. Train: {len(train_ids)}, Val: {len(val_ids)}")


In [ ]:
# 2. CHUYỂN ĐỔI COCO SANG YOLO
def convert_coco_to_yolo(bbox, img_width, img_height):
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return x_center, y_center, w_norm, h_norm

print("Đang xử lý và tạo file .txt cho RT-DETR...")

def process_split(ids, split_name):
    for img_id in tqdm(ids, desc=f"Processing {split_name}"):
        img_info = images_info[img_id]
        img_filename = img_info['file_name']
        img_width = img_info['width']
        img_height = img_info['height']
        
        src_img_path = os.path.join(image_dir, img_filename)
        dst_img_path = os.path.join(dataset_dir, split_name, 'images', img_filename)
        
        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
            
            txt_filename = img_filename.rsplit('.', 1)[0] + '.txt'
            txt_path = os.path.join(dataset_dir, split_name, 'labels', txt_filename)
            
            with open(txt_path, 'w') as f_txt:
                if img_id in img_to_anns:
                    for ann in img_to_anns[img_id]:
                        class_id = int(ann['category_id']) - 1
                        x_c, y_c, w_n, h_n = convert_coco_to_yolo(ann['bbox'], img_width, img_height)
                        f_txt.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

process_split(train_ids, 'train')
process_split(val_ids, 'val')

print("Hoàn tất quá trình chuẩn bị dữ liệu!")


In [ ]:
# 3. TẠO FILE CẤU HÌNH DATASET.YAML
yaml_content = f"""
path: {dataset_dir}
train: train/images
val: val/images

names:
  0: No entry
  1: No parking / waiting
  2: No turning
  3: Max Speed
  4: Other prohibition signs
  5: Warning signs
  6: Mandatory signs
"""

with open('/kaggle/working/dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())
    
print("Đã tạo xong file cấu hình dataset.yaml")


In [ ]:
# 4. CÀI ĐẶT MÔI TRƯỜNG
!pip install -q ultralytics
import ultralytics
ultralytics.checks()


In [ ]:
# 5. HUẤN LUYỆN MÔ HÌNH RT-DETR-L
from ultralytics import RTDETR

# Khởi tạo mô hình RT-DETR bản Large (Transformer gốc)
model = RTDETR('rtdetr-l.pt')

# Bắt đầu huấn luyện với cấu hình CHỐNG TRÀN RAM (Gradient Accumulation)
results = model.train(
    data='/kaggle/working/dataset.yaml',
    epochs=50,
    imgsz=1280,         # [TỪ E3] Bắt buộc giữ độ phân giải cao cho vật thể nhỏ
    batch=2,            # [QUAN TRỌNG] Ép batch=2 để GPU Kaggle (T4/P100) không chết ngạt VRAM
    accumulate=4,       # Tích lũy 4 mini-batch -> Batch ảo = 2x4 = 8
    optimizer='AdamW',  # Thuật toán chuẩn công nghiệp cho mạng Transformer
    cos_lr=True,        # Hạ nhiệt độ Learning Rate
    project='/kaggle/working/zalo_traffic', 
    name='rtdetr_highres',
    device=0,
)


In [ ]:
# 6. ĐÓNG GÓI VÀ ÉP TRÌNH DUYỆT TỰ TẢI VỀ (AUTO-DOWNLOAD)
from IPython.display import HTML

print("Đang nén toàn bộ kết quả vào file zip...")
!zip -r -q /kaggle/working/rtdetr_results_ZaloAI.zip /kaggle/working/zalo_traffic
print("Đã nén xong! Trình duyệt sẽ tự động tải file về ngay bây giờ...")

# Dùng JavaScript tạo một thẻ <a> ẩn và tự động click vào nó
html_code = """
<a id="auto_download" href="rtdetr_results_ZaloAI.zip" download>Đang tải xuống...</a>
<script>
    document.getElementById("auto_download").click();
</script>
"""
HTML(html_code)


In [ ]:
# 7. [TỪ E3, M4.2] THUẬT TOÁN SAHI (Sẵn sàng cho Web App)
!pip install -q sahi
from sahi import AutoDetectionModel

try:
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',  # SAHI load Ultralytics RTDETR thông qua type 'yolov8'
        model_path='/kaggle/working/zalo_traffic/rtdetr_highres/weights/best.pt',
        confidence_threshold=0.25,
        device="cuda:0"
    )
    print("Hệ thống SAHI đã sẵn sàng cho RT-DETR trên Web App.")
except Exception as e:
    print("Lưu ý: Chỉ chạy được SAHI sau khi đã huấn luyện xong (có file best.pt):", e)
